<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0 Fixed Grid Backtest

Single source of truth: configuration, backtest engine, audit, and GitHub log export all live in this notebook.

V0 uses user-defined **Initial Capital, Floor, Ceiling, Gap, and Fees**. Floor/Ceiling are not derived from future historical High/Low.

## 1. Setup & User Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import heapq
import bisect
import json
import base64
import requests
import numpy as np
import pandas as pd

# =========================
# USER CONFIGURATION
# =========================
DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'
SYMBOL = 'BTCUSDT'
TIMEFRAME = '1m'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'

INITIAL_CAPITAL = 3000.0
GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0
BUY_FEE = 0.001
SELL_FEE = 0.001

REPO = 'natdanaiii/Trading'
BRANCH = 'main'
GITHUB_LOG_PATH = 'logs/latest_v0_backtest_log.json'
LOCAL_LOG_PATH = '/content/latest_v0_backtest_log.json'

## 2. V0 Engine

In [ ]:
def load_market_data(symbol, timeframe, data_dir):
    path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    required = {'open_time','open','high','low','close','volume'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    df['open_time'] = pd.to_datetime(df['open_time'], utc=True)
    numeric = ['open','high','low','close','volume']
    df[numeric] = df[numeric].astype(float)
    start_ts = pd.Timestamp(START_DATE, tz='UTC')
    end_ts = pd.Timestamp(END_DATE, tz='UTC')
    return (df.drop_duplicates('open_time').sort_values('open_time')
            .loc[lambda x: (x['open_time'] >= start_ts) & (x['open_time'] < end_ts)]
            .reset_index(drop=True))

def build_fixed_grid_table(capital, floor, ceiling, gap, buy_fee, sell_fee):
    if capital <= 0 or floor <= 0 or ceiling <= floor or gap <= 0:
        raise ValueError('Invalid grid configuration.')
    if not (0 <= buy_fee < 1 and 0 <= sell_fee < 1):
        raise ValueError('fees must be in [0, 1).')
    raw_count = (ceiling - floor) / gap
    if not np.isclose(raw_count, round(raw_count)):
        raise ValueError('(ceiling - floor) must be exactly divisible by gap.')
    n = int(round(raw_count))
    capital_per_grid = capital / n
    buy_prices = ceiling - gap * np.arange(1, n + 1)
    sell_prices = buy_prices + gap
    gross_base = capital_per_grid / buy_prices
    buy_fee_base = gross_base * buy_fee
    base_amount = gross_base - buy_fee_base
    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_grid
    return pd.DataFrame({
        'level': np.arange(1, n + 1), 'buy_price': buy_prices,
        'sell_price': sell_prices, 'capital_per_grid': capital_per_grid,
        'base_amount': base_amount, 'buy_fee_base': buy_fee_base,
        'sell_fee_quote': sell_fee_quote, 'net_sell': net_sell, 'profit': profit
    })

def performance_stats(data, equity, initial_capital):
    running_peak = np.maximum.accumulate(equity)
    drawdown = equity / running_peak - 1.0
    final_equity = float(equity[-1])
    max_drawdown = float(drawdown.min())
    net_return = final_equity / initial_capital - 1.0
    elapsed_days = (data['open_time'].iloc[-1] - data['open_time'].iloc[0]).total_seconds() / 86400.0
    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        growth = np.log(final_equity / initial_capital) * (365.25 / elapsed_days)
        if growth < 700:
            annualized_return = float(np.expm1(growth))
    calmar = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar = float(annualized_return / abs(max_drawdown))
    return final_equity, net_return, annualized_return, max_drawdown, calmar, drawdown

def run_fixed_grid_backtest(df_price, grid, initial_capital):
    data = df_price.sort_values('open_time').reset_index(drop=True)
    grid = grid.sort_values('buy_price').reset_index(drop=True).copy()
    if data.empty or grid.empty:
        raise ValueError('Market data and grid must not be empty.')
    buy_prices = grid['buy_price'].to_numpy(float)
    sell_prices = grid['sell_price'].to_numpy(float)
    costs = grid['capital_per_grid'].to_numpy(float)
    base_amounts = grid['base_amount'].to_numpy(float)
    buy_fee_base = grid['buy_fee_base'].to_numpy(float)
    sell_fee_quote = grid['sell_fee_quote'].to_numpy(float)
    net_sell = grid['net_sell'].to_numpy(float)
    cycle_profit = grid['profit'].to_numpy(float)
    holding = np.zeros(len(grid), dtype=bool)
    buy_time = [None] * len(grid)
    sell_heap = []
    cash, btc, realized_profit = float(initial_capital), 0.0, 0.0
    total_buy_fee_usdt = total_sell_fee_usdt = 0.0
    completed_cycles = 0
    events, completed = [], []
    equity_values = np.empty(len(data)); cash_values = np.empty(len(data)); btc_values = np.empty(len(data))
    buy_price_list = buy_prices.tolist()
    previous_close = None
    event_id = 0
    for i, row in enumerate(data.itertuples(index=False)):
        timestamp = row.open_time
        open_price, high_price, low_price, close_price = map(float, (row.open,row.high,row.low,row.close))
        cash_at_candle_start = cash
        sold_this_candle = set()
        while sell_heap and sell_heap[0][0] <= high_price:
            _, k = heapq.heappop(sell_heap)
            if not holding[k]:
                continue
            cash_before, btc_before = cash, btc
            holding[k] = False
            cash += net_sell[k]; btc -= base_amounts[k]
            if abs(btc) < 1e-12: btc = 0.0
            realized_profit += cycle_profit[k]
            total_sell_fee_usdt += sell_fee_quote[k]
            completed_cycles += 1
            sold_this_candle.add(k); event_id += 1
            completed.append({'buy_time':buy_time[k],'sell_time':timestamp,'buy_price':buy_prices[k],'sell_price':sell_prices[k],'cost':costs[k],'profit':cycle_profit[k]})
            events.append({'event_id':event_id,'time':timestamp,'side':'SELL','price':sell_prices[k],'cash_movement':net_sell[k],'grid_cashflow':cycle_profit[k],'cash_before':cash_before,'cash_after':cash,'btc_before':btc_before,'btc_after':btc})
            buy_time[k] = None
        buy_budget = cash_at_candle_start
        downward_start = open_price if previous_close is None else max(previous_close, open_price)
        if low_price < downward_start:
            first_index = bisect.bisect_left(buy_price_list, low_price)
            stop_index = bisect.bisect_left(buy_price_list, downward_start)
            for k in range(stop_index - 1, first_index - 1, -1):
                if holding[k] or k in sold_this_candle:
                    continue
                if buy_budget + 1e-12 < costs[k]:
                    break
                cash_before, btc_before = cash, btc
                holding[k] = True; buy_time[k] = timestamp
                buy_budget -= costs[k]; cash -= costs[k]; btc += base_amounts[k]
                total_buy_fee_usdt += buy_fee_base[k] * buy_prices[k]
                heapq.heappush(sell_heap, (sell_prices[k], k)); event_id += 1
                events.append({'event_id':event_id,'time':timestamp,'side':'BUY','price':buy_prices[k],'cash_movement':-costs[k],'grid_cashflow':0.0,'cash_before':cash_before,'cash_after':cash,'btc_before':btc_before,'btc_after':btc})
        equity_values[i] = cash + btc * close_price
        cash_values[i] = cash; btc_values[i] = btc; previous_close = close_price
    final_equity, net_return, annualized_return, max_drawdown, calmar, drawdown = performance_stats(data, equity_values, initial_capital)
    equity_curve = pd.DataFrame({'open_time':data['open_time'],'close':data['close'],'cash':cash_values,'btc':btc_values,'equity':equity_values,'drawdown':drawdown})
    trade_log = pd.DataFrame(events)
    summary = {'initial_capital':initial_capital,'final_equity':final_equity,'net_return':net_return,'annualized_return':annualized_return,'max_drawdown':max_drawdown,'calmar_ratio':calmar,'completed_cycles':completed_cycles,'open_positions':int(holding.sum()),'final_cash':cash,'final_btc':btc,'realized_profit':realized_profit,'unrealized_pnl':final_equity-initial_capital-realized_profit,'total_fee_usdt_equiv':total_buy_fee_usdt+total_sell_fee_usdt}
    return {'summary':summary,'trade_log':trade_log,'completed_trades':pd.DataFrame(completed),'equity_curve':equity_curve,'grid_state':grid.assign(holding=holding,buy_time=buy_time)}

def audit_v0(result):
    summary, log, equity = result['summary'], result['trade_log'], result['equity_curve']
    cash_movement = log['cash_movement'].sum() if len(log) else 0.0
    grid_cashflow = log['grid_cashflow'].sum() if len(log) else 0.0
    return {
        'cash_reconciliation': abs(INITIAL_CAPITAL + cash_movement - summary['final_cash']) <= 1e-8,
        'realized_profit_reconciliation': abs(grid_cashflow - summary['realized_profit']) <= 1e-8,
        'equity_identity': float(np.max(np.abs(equity['cash'] + equity['btc']*equity['close'] - equity['equity']))) <= 1e-8,
        'cash_never_negative': float(equity['cash'].min()) >= -1e-8
    }

## 3. Run Backtest & Audit

In [ ]:
df_1m = load_market_data(SYMBOL, TIMEFRAME, DATA_DIR)
grid = build_fixed_grid_table(INITIAL_CAPITAL, GRID_FLOOR, GRID_CEILING, GRID_GAP, BUY_FEE, SELL_FEE)
result = run_fixed_grid_backtest(df_1m, grid, INITIAL_CAPITAL)
summary = result['summary']
audit_checks = audit_v0(result)
AUDIT_STATUS = 'PASS' if all(audit_checks.values()) else 'FAIL'

print('===== V0 CONFIGURATION =====')
print(f'Capital         : {INITIAL_CAPITAL:,.2f} USDT')
print(f'Floor           : {GRID_FLOOR:,.2f} USDT')
print(f'Ceiling         : {GRID_CEILING:,.2f} USDT')
print(f'Gap             : {GRID_GAP:,.2f} USDT')
print(f'Number of Grids : {len(grid)}')
print(f"Capital / Grid  : {grid['capital_per_grid'].iloc[0]:,.6f} USDT")
print('\n===== V0 RESULT =====')
for k in ['final_equity','net_return','annualized_return','max_drawdown','calmar_ratio','completed_cycles','open_positions','final_cash','final_btc','realized_profit','unrealized_pnl','total_fee_usdt_equiv']:
    print(f'{k:24s}: {summary[k]}')
print('\n===== V0 AUDIT =====')
for name, passed in audit_checks.items():
    print(f"{name:32s}: {'PASS' if passed else 'FAIL'}")
print(f'Overall                         : {AUDIT_STATUS}')
if AUDIT_STATUS != 'PASS':
    raise AssertionError('V0 AUDIT FAILED')

## 4. Export Latest Log to GitHub

In [ ]:
def json_safe(value):
    if value is pd.NaT or value is pd.NA: return None
    if isinstance(value, dict): return {str(k): json_safe(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray): return [json_safe(v) for v in value.tolist()]
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if isinstance(value, (int, np.integer)): return int(value)
    if isinstance(value, (float, np.floating)): return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp): return value.isoformat()
    return value

log_payload = {
    'log_schema_version': 2, 'strategy': 'V0 Fixed Grid',
    'run_info': {'generated_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),'repository':REPO,'branch':BRANCH,'notebook':'Grid_trading_V0.ipynb','symbol':SYMBOL,'timeframe':TIMEFRAME,'start_date':START_DATE,'end_date':END_DATE,'data_rows':int(len(df_1m)),'data_first_time':df_1m['open_time'].min().isoformat(),'data_last_time':df_1m['open_time'].max().isoformat()},
    'parameters': {'initial_capital':INITIAL_CAPITAL,'floor':GRID_FLOOR,'ceiling':GRID_CEILING,'gap':GRID_GAP,'buy_fee':BUY_FEE,'sell_fee':SELL_FEE},
    'derived': {'number_of_grids':int(len(grid)),'capital_per_grid':float(grid['capital_per_grid'].iloc[0])},
    'market_range_diagnostics': {'historical_low':float(df_1m['low'].min()),'historical_high':float(df_1m['high'].max()),'candles_low_below_floor':int((df_1m['low'] < GRID_FLOOR).sum()),'candles_high_above_ceiling':int((df_1m['high'] > GRID_CEILING).sum())},
    'summary': summary, 'audit': {'status':AUDIT_STATUS,'checks':audit_checks}
}
log_payload = json_safe(log_payload)
with open(LOCAL_LOG_PATH,'w',encoding='utf-8') as f:
    json.dump(log_payload,f,indent=2,allow_nan=False)

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' was not found.")
else:
    api_url = f'https://api.github.com/repos/{REPO}/contents/{GITHUB_LOG_PATH}'
    headers = {'Authorization':f'Bearer {github_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
    existing = requests.get(api_url,headers=headers,timeout=30)
    body = {'message':'Update latest V0 backtest log','content':base64.b64encode(json.dumps(log_payload,indent=2,allow_nan=False).encode()).decode(),'branch':BRANCH}
    if existing.status_code == 200: body['sha'] = existing.json()['sha']
    upload = requests.put(api_url,headers=headers,json=body,timeout=30); upload.raise_for_status()
    print('GitHub log upload: SUCCESS')
    print(f'Path: {GITHUB_LOG_PATH}')
    print(f"Commit SHA: {upload.json()['commit']['sha']}")